# 02. Preprocessing & Feature Engineering

This notebook details the preprocessing and feature engineering steps. In time series forecasting, we extract temporal features (Year, Month, etc.) and create historical features like lags and rolling statistics to capture autocorrelation, ensuring we group by `store` and `item` to avoid leakage.

In [ ]:
import pandas as pd
import numpy as np

# Load raw data
df = pd.read_csv("../data/train.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['store', 'item', 'date']).reset_index(drop=True)
df.head()

## 1. Extract Time Features

In [ ]:
def extract_time_features(data):
    df = data.copy()
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['week'] = df['date'].dt.isocalendar().week.astype(int)
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['quarter'] = df['date'].dt.quarter
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    return df

df_time = extract_time_features(df)
df_time.head()

## 2. Generate Lag Features

Lag features help our model learn past sales behavior. We will create lags of 7, 14, and 30 days.

In [ ]:
def add_lags(df, lags=[7, 14, 30]):
    for lag in lags:
        df[f'sales_lag_{lag}'] = df.groupby(['store', 'item'])['sales'].shift(lag)
    return df

df_lags = add_lags(df_time)
df_lags[['date', 'store', 'item', 'sales', 'sales_lag_7', 'sales_lag_14', 'sales_lag_30']].head(10)

## 3. Generate Rolling Mean Features

Rolling averages smooth out noise and capture recent trends. We compute 7, 14, and 30 days rolling means.

In [ ]:
def add_rolling_means(df, windows=[7, 14, 30]):
    for window in windows:
        # In time series forecasting, we shift by 1 before calculating rolling statistics
        # to avoid data leakage (knowing today's sales to predict today's sales)
        df[f'sales_roll_mean_{window}'] = df.groupby(['store', 'item'])['sales'] \
                                              .shift(1) \
                                              .transform(lambda x: x.rolling(window=window).mean())
    return df

df_rolling = add_rolling_means(df_lags)
df_rolling[['date', 'store', 'item', 'sales', 'sales_roll_mean_7', 'sales_roll_mean_14', 'sales_roll_mean_30']].head(12)

## 4. Handling NaN Values

Shifting and rolling create `NaN` values at the beginning of each store-item timeline. We will drop these rows.

In [ ]:
print(f"Shape before dropping NaNs: {df_rolling.shape}")
df_final = df_rolling.dropna().reset_index(drop=True)
print(f"Shape after dropping NaNs: {df_final.shape}")
df_final.head()